In [ ]:
# 1) Baixa o repositório e instala (no Kaggle: ative "Internet" nas configurações do notebook)
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Kaua-Martins-Santos/delivery-agent"
cwd = Path.cwd()
if (cwd / "pyproject.toml").exists():                       # já dentro do repo
    REPO = cwd
elif cwd.name == "notebooks" and (cwd.parent / "pyproject.toml").exists():
    REPO = cwd.parent                                       # rodando de notebooks/
else:
    REPO = cwd / "delivery-agent"
    if not REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[embeddings]"], check=True)
print("Repo:", REPO)

# Benchmark de embeddings para RAG multi-tenant

**O problema.** O `delivery-agent` atende vários restaurantes com uma única base vetorial. Cada
pergunta ("até que horas fica aberto?") precisa recuperar o trecho certo **do restaurante certo**,
sem vazar nada de outro. A qualidade da busca depende do modelo de embeddings.

Aqui comparamos quatro embedders nos 40 casos do `evals/datasets/golden.jsonl`, com o LLM **fake**
(extrativo, sem custo), para isolar o efeito da recuperação:

| Embedder | O que é | Dim. |
|---|---|---|
| `hash` | n-gramas com hashing, sem modelo (baseline do CI) | 384 |
| `local` | paraphrase-multilingual-MiniLM-L12-v2 | 384 |
| `e5-small` | intfloat/multilingual-e5-small | 384 |
| `e5-base` | intfloat/multilingual-e5-base | **768** |

> **Atenção:** o `e5-base` tem 768 dimensões e **não cabe** na coluna `vector(384)` do Postgres
> (`db/init.sql`). Aqui ele roda só no `InMemoryStore`, que aceita qualquer dimensão. Serve como
> comparação; para usá-lo em produção seria preciso mudar a coluna e reindexar.

## O que cada métrica mede

- **Acerto**: % de casos em que a resposta final está certa (e sem vazamento).
- **Recall@k**: % de casos em que o trecho esperado veio entre os `k=4` recuperados. Mede **só a busca**.
- **Vazamentos**: respostas com termos de outro restaurante. Precisa ser **0**.
- **Latência p95**: tempo da resposta no percentil 95. Com 40 casos ela oscila muito, então cada
  embedder roda **3 vezes** e usamos a **média** (com desvio-padrão).

In [ ]:
# 2) Configuração
import json, os, tempfile, time
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "sem GPU")

EMBEDDERS = {  # nome -> variáveis de ambiente
    "hash":     {"EMBEDDER": "hash"},
    "local":    {"EMBEDDER": "local"},
    "e5-small": {"EMBEDDER": "e5", "E5_MODEL": "intfloat/multilingual-e5-small"},
    "e5-base":  {"EMBEDDER": "e5", "E5_MODEL": "intfloat/multilingual-e5-base"},
}
RUNS = 3
OUT = Path(tempfile.mkdtemp(prefix="bench_emb_"))

### (Opcional) O dataset no Hugging Face Hub

Os 40 casos também estão publicados em
[Martinsk123/delivery-agent-evals](https://huggingface.co/datasets/Martinsk123/delivery-agent-evals).
A célula abaixo carrega direto do Hub e confere se é igual ao arquivo do repositório. Os evals
continuam lendo o `golden.jsonl` local; esta célula é só para explorar o dataset.

In [ ]:
# (Opcional) Carrega o dataset de evals direto do Hub
try:
    from datasets import load_dataset
    ds = load_dataset("Martinsk123/delivery-agent-evals", split="test")
    local = [json.loads(l)["id"] for l in (REPO / "evals/datasets/golden.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
    print(ds)
    print("Mesmos casos do arquivo local:", sorted(ds["id"]) == sorted(local))
    display(ds.to_pandas()["categoria"].value_counts().rename("casos").to_frame())
except Exception as exc:  # sem internet ou sem a biblioteca: não atrapalha o resto do notebook
    print("Pulando (opcional):", type(exc).__name__, exc)

In [ ]:
# 3) Roda os evals (LLM fake) RUNS vezes por embedder
def run_eval(name: str, i: int) -> dict:
    out = OUT / f"{name}-{i}"
    env = {**os.environ, "LLM_PROVIDER": "fake", "PYTHONIOENCODING": "utf-8", **EMBEDDERS[name]}
    p = subprocess.run([sys.executable, "-m", "evals.run_evals", "--out", str(out)],
                       cwd=REPO, env=env, capture_output=True, text=True, encoding="utf-8")
    report = next(out.glob("latest-*.json"), None)
    if report is None:
        raise RuntimeError(f"{name}: evals falharam\n{p.stderr[-1500:]}")
    return json.loads(report.read_text(encoding="utf-8"))

results = {}
for name in EMBEDDERS:
    t0 = time.time()
    results[name] = [run_eval(name, i) for i in range(RUNS)]
    p95 = [r["resumo"]["latencia_ms_p95"] for r in results[name]]
    print(f"{name:9} p95 por rodada: {p95} ms  ({time.time() - t0:.0f}s)")

## Resultado

In [ ]:
# 4) Tabela
import pandas as pd

rows = []
for name, runs in results.items():
    s = runs[0]["resumo"]                       # acerto e recall são determinísticos entre rodadas
    p95 = pd.Series([r["resumo"]["latencia_ms_p95"] for r in runs])
    faq20 = next(c for c in runs[0]["casos"] if c["id"] == "faq-20")
    rows.append({
        "embedder": name,
        "acerto (%)": round(s["taxa_acerto"] * 100, 1),
        "recall@k (%)": round(s["recall_at_k_recuperacao"] * 100, 1),
        "vazamentos": s["casos_com_vazamento"],
        "p95 médio (ms)": round(p95.mean(), 1),
        "p95 desvio (ms)": round(p95.std(), 1),
        "faq-20: fonte certa?": "sim" if faq20["checks"]["fonte"] else "não",
        "faq-20: passou?": "sim" if faq20["passou"] else "não",
    })
df = pd.DataFrame(rows).set_index("embedder")
df

In [ ]:
# 5) Gráfico: acerto e recall@k por embedder
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(df)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(x - w/2, df["acerto (%)"], w, label="Acerto")
b2 = ax.bar(x + w/2, df["recall@k (%)"], w, label="Recall@k")
ax.bar_label(b1, fmt="%.0f%%", padding=2); ax.bar_label(b2, fmt="%.0f%%", padding=2)
ax.set_xticks(x, df.index); ax.set_ylim(0, 110); ax.set_ylabel("%")
ax.set_title("Embeddings no RAG do delivery-agent (40 casos, LLM fake)")
ax.legend(loc="lower right"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 6) O caso faq-20 ("até que horas fica aberto?")
for name, runs in results.items():
    c = next(c for c in runs[0]["casos"] if c["id"] == "faq-20")
    print(f"{name:9} fonte certa: {'sim' if c['checks']['fonte'] else 'não':3} | passou: {'sim' if c['passou'] else 'não':3} | {c['resposta'][:70]}")

## Conclusão

- **Recall@k mede a busca; acerto mede a resposta inteira.** Os dois só andam juntos quando o LLM
  consegue usar o que foi recuperado.
- **faq-20:** o `hash` não recupera o trecho do horário. Os modelos semânticos recuperam (veja a
  célula acima), mas o caso continua reprovado porque o **LLM fake** escolhe as frases por
  hashing e não acha a do horário. **O problema agora é o LLM, não a busca.** Só um LLM de verdade
  deve fechar esse caso.
- **Vazamentos:** devem ser 0 em todos. O isolamento vem do `tenant_id` na busca (e do RLS no
  Postgres), não do embedder.
- **Latência:** modelos semânticos custam dezenas de ms a mais que o `hash`; por isso a média de 3 rodadas.
  Em GPU os números mudam.
- **e5-base (768 dim)** é só comparação: não cabe no Postgres atual. Neste dataset ele **não
  melhorou sobre o e5-small** (mesmo acerto e recall, cerca de 2x a latência), então o **e5-small é a escolha**.
- Para melhorar de verdade: rodar com um LLM real (`--judge`) e ampliar o dataset.